# Notebook 13 — Multiplicative Inverses in GF(2⁸)

**Volume 1: Foundations, Finite Fields, and AES**

## Sections
1. Introduction and setup
2. GF(2⁸) multiply — xtime chain approach (reused from Module 12)
3. Brute-force inverse finder
4. Check a known pair: 0x02 and 0x8D
5. Check the AES S-box pair: 0x53 and 0xCA
6. Verify all 255 inverses are correct and unique
7. Interactive inverse explorer — `gf_inv_explore(hex_val)`
8. Special cases — 0x00, 0x01, self-inverse bytes
9. Connection to the AES S-box
10. Summary table of sample inverse pairs

## Section 1 — Introduction

Module 12 built the machinery for GF(2⁸) multiplication.  
This module asks the natural follow-up question:

> **For a nonzero byte `a`, is there always a byte `b` such that `a * b == 0x01`?**

The answer is **yes** — because the AES modulus

    m(x) = x^8 + x^4 + x^3 + x + 1   (hex: 0x11B)

is *irreducible* over GF(2).  Working modulo an irreducible polynomial turns the quotient
ring into a **field**, and in any field every nonzero element has a multiplicative inverse.

### Why it matters for AES

The first stage of the **AES SubBytes (S-box)** operation is exactly this:
replace each nonzero byte with its GF(2⁸) inverse.  Zero maps to zero.  
This non-linear step is the primary source of confusion and diffusion in AES.

### Plan for this notebook

We will:
- Re-use `gf_mul` from Module 12 (self-contained here)
- Write a brute-force `gf_inv` that searches all 255 candidates
- Verify two well-known inverse pairs
- Confirm that the 255 nonzero inverses are all distinct (forming a permutation)
- Explore special and self-inverse bytes
- Connect the inverse table to the AES S-box

## Section 2 — GF(2⁸) multiply — xtime chain approach

We re-declare the multiply functions here so this notebook is **self-contained**.

**Representation:** A polynomial over GF(2) is stored as an integer bitmask.  
Bit `d` = coefficient of `x^d`; LSB = constant term.  
Example: `0x57 = 0101 0111` encodes `x^6 + x^4 + x^2 + x + 1`.

**xtime rule:**
- Left-shift by 1 (= multiply by `x`).
- If the original top bit (bit 7) was 1, XOR with `0x1B` to fold the `x^8` term back using `x^8 ≡ x^4 + x^3 + x + 1`.

**gf_mul:** build the xtime chain `a, xt(a), xt²(a), …` then XOR the entries selected by the set bits of `b`.

In [ ]:
# Section 2 — GF(2^8) multiplication helpers
# (Same approach as Module 12 — re-declared for self-containment.)
#
# Polynomial bitmask convention:
#   bit d  =  coefficient of x^d   (LSB = constant term)

def xt(a):
    """Multiply byte a by x (= 0x02) in GF(2^8) — the xtime operation."""
    s = (a << 1) & 0xFF
    return (s ^ 0x1B) if (a & 0x80) else s

def gf_mul(a, b):
    """Multiply bytes a and b in GF(2^8) using the xtime chain."""
    r, aa = 0, a
    for i in range(8):
        if (b >> i) & 1:
            r ^= aa
        aa = xt(aa)
    return r

# Smoke-test: known GF(2^8) products from Module 12
smoke = [
    (0x57, 0x01, 0x57),   # a * 1 = a
    (0x57, 0x02, 0xAE),   # xtime(0x57)
    (0x57, 0x13, 0xFE),   # worked example from tutorial
    (0x83, 0x02, 0x1D),   # xtime with reduction
    (0x00, 0xFF, 0x00),   # zero annihilator
]
print('gf_mul smoke tests:')
for a, b, expected in smoke:
    result = gf_mul(a, b)
    status = '✓' if result == expected else f'✗ expected 0x{expected:02X}'
    print(f'  0x{a:02X} × 0x{b:02X} = 0x{result:02X}  {status}')

## Section 3 — Brute-force inverse finder

The simplest way to find `a⁻¹` is to try every candidate `b` from 1 to 255 and check
whether `gf_mul(a, b) == 1`.  Because GF(2⁸) is a field, exactly one such `b` exists
for each nonzero `a`.

Zero has no inverse — no byte multiplied by zero can ever yield `0x01` — so we return
`None` for `a = 0`.

In [ ]:
# Section 3 — Brute-force inverse finder

def gf_inv(a):
    """
    Return the multiplicative inverse of byte a in GF(2^8).

    Parameters
    ----------
    a : int  — byte value (0..255)

    Returns
    -------
    int  — the inverse b such that gf_mul(a, b) == 1, or None if a == 0.
    """
    if a == 0:
        return None
    for b in range(1, 256):
        if gf_mul(a, b) == 1:
            return b
    return None   # never reached for a nonzero element of GF(2^8)

# Quick demonstration with a handful of values
demo_bytes = [0x01, 0x02, 0x03, 0x53, 0x83, 0xFF, 0x1A, 0x7E]
print(f'{"Byte":>6}  {"Inverse":>8}  {"Product (verify)":>18}')
print('-' * 42)
for a in demo_bytes:
    inv = gf_inv(a)
    if inv is None:
        print(f'0x{a:02X}     None       (no inverse)')
    else:
        prod = gf_mul(a, inv)
        check = '✓' if prod == 1 else '✗'
        print(f'0x{a:02X}     0x{inv:02X}      0x{prod:02X}  {check}')

## Section 4 — Check a known pair: 0x02 and 0x8D

One of the simplest nontrivial inverse pairs is `(0x02, 0x8D)`.
We verify it from both directions and show every step of the multiplication.

In [ ]:
# Section 4 — Verify the pair (0x02, 0x8D)

a, b = 0x02, 0x8D

def gf_mul_trace(a, b):
    """Multiply a × b in GF(2^8), printing the xtime chain and selected terms."""
    print(f'Computing 0x{a:02X} × 0x{b:02X}')
    print(f'  B = 0x{b:02X} = {b:08b}b')
    print()
    chain = []
    aa = a
    for i in range(8):
        chain.append(aa)
        aa = xt(aa)
    print('  xtime chain for A:')
    selected = []
    for i, v in enumerate(chain):
        flag = '  <-- selected' if (b >> i) & 1 else ''
        print(f'    x^{i}·A = 0x{v:02X}{flag}')
        if (b >> i) & 1:
            selected.append((i, v))
    print()
    result = 0
    parts = []
    for i, v in selected:
        result ^= v
        parts.append(f'0x{v:02X}')
    xor_str = ' ⊕ '.join(parts)
    print(f'  {xor_str} = 0x{result:02X}')
    return result

print('=== Direction 1: 0x02 × 0x8D ===')
r1 = gf_mul_trace(a, b)
print()
print('=== Direction 2: 0x8D × 0x02 ===')
r2 = gf_mul_trace(b, a)
print()
print(f'0x{a:02X} × 0x{b:02X} = 0x{r1:02X}  (expected 0x01: {"✓" if r1 == 1 else "✗"})')
print(f'0x{b:02X} × 0x{a:02X} = 0x{r2:02X}  (expected 0x01: {"✓" if r2 == 1 else "✗"})')
print()
print(f'gf_inv(0x02) = 0x{gf_inv(0x02):02X}  (expected 0x8D: {"✓" if gf_inv(0x02) == 0x8D else "✗"})')

## Section 5 — Check the AES S-box pair: 0x53 and 0xCA

The AES specification uses the inverse of `0x53` as its worked example for the S-box:
the inverse is `0xCA`.  We verify this and then apply the full S-box affine transform
to reproduce the documented S-box output `0xED`.

In [ ]:
# Section 5 — The AES S-box worked example: 0x53

a = 0x53
inv_a = gf_inv(a)
prod = gf_mul(a, inv_a)

print(f'Input byte     : 0x{a:02X} = {a:08b}b')
print(f'GF(2^8) inverse: 0x{inv_a:02X} = {inv_a:08b}b')
print(f'Verification   : 0x{a:02X} × 0x{inv_a:02X} = 0x{prod:02X}  ({"✓" if prod == 1 else "✗"})')
print()
print(f'Expected from AES spec: 0x53 → inverse = 0xCA')
print(f'Our result matches    : {inv_a == 0xCA}')
print()

# AES S-box affine transform (FIPS 197, Section 5.1.1)
# b_i = a_i XOR a_{i+4 mod 8} XOR a_{i+5 mod 8} XOR a_{i+6 mod 8} XOR a_{i+7 mod 8} XOR c_i
# where c = 0x63 = 0110 0011b
def aes_affine(byte_val):
    """Apply the AES affine transform to a byte (after taking the GF(2^8) inverse)."""
    b = byte_val
    result = 0
    c = 0x63  # affine constant
    for i in range(8):
        # bit i of result = bit i of b XOR bit (i+4)%8 XOR bit (i+5)%8 XOR bit (i+6)%8 XOR bit (i+7)%8 XOR bit i of c
        bit = ((b >> i) & 1) ^ ((b >> ((i + 4) % 8)) & 1) ^ \
              ((b >> ((i + 5) % 8)) & 1) ^ ((b >> ((i + 6) % 8)) & 1) ^ \
              ((b >> ((i + 7) % 8)) & 1) ^ ((c >> i) & 1)
        result |= (bit << i)
    return result

sbox_out = aes_affine(inv_a)
print(f'Affine transform of 0x{inv_a:02X} (= inverse of 0x{a:02X})')
print(f'  S-box output: 0x{sbox_out:02X}')
print(f'  Expected (FIPS 197 Table 5): 0xED')
print(f'  Matches: {sbox_out == 0xED}')

## Section 6 — Verify all 255 inverses are correct and unique

In a field, the map `a ↦ a⁻¹` is a **bijection** on the nonzero elements.  
This means:
- Every nonzero byte has an inverse (surjectivity).
- No two distinct nonzero bytes share the same inverse (injectivity).

We check both properties exhaustively.

In [ ]:
# Section 6 — Exhaustive verification of all 255 inverses

print('Building the full inverse table for bytes 0x01 .. 0xFF ...')
inv_table = {}  # a -> gf_inv(a)
errors = []

for a in range(1, 256):
    b = gf_inv(a)
    # Confirm b is not None and gf_mul(a, b) == 1
    if b is None:
        errors.append(f'  gf_inv(0x{a:02X}) returned None!')
    elif gf_mul(a, b) != 1:
        errors.append(f'  gf_mul(0x{a:02X}, 0x{b:02X}) = 0x{gf_mul(a,b):02X} != 0x01!')
    else:
        inv_table[a] = b

if errors:
    print('ERRORS found:')
    for e in errors: print(e)
else:
    print(f'  All 255 nonzero bytes have a valid inverse. ✓')

# Check uniqueness: the set of all inverse values should have 255 elements
inv_values = list(inv_table.values())
unique_inv = set(inv_values)
print(f'  Number of distinct inverse values: {len(unique_inv)} (expected 255)')
print(f'  Injectivity (bijection): {len(unique_inv) == 255}  ✓' if len(unique_inv) == 255 else '  ✗ NOT a bijection!')

# Confirm involution: (a^-1)^-1 == a
involution_ok = all(gf_inv(gf_inv(a)) == a for a in range(1, 256))
print(f'  Involution: (a⁻¹)⁻¹ = a for all nonzero a: {involution_ok}  ✓' if involution_ok else '  Involution FAILED!')

# Count self-inverse bytes (a == a^-1, i.e., a^2 == 0x01)
self_inv = [a for a in range(1, 256) if inv_table[a] == a]
print(f'\nSelf-inverse bytes (a⁻¹ = a): {len(self_inv)} found')
for v in self_inv:
    print(f'  0x{v:02X} : gf_mul(0x{v:02X}, 0x{v:02X}) = 0x{gf_mul(v,v):02X}')

## Section 7 — Interactive inverse explorer

`gf_inv_explore(hex_val)` accepts a hex string like `"0x53"` or a plain integer,
and prints a full breakdown:
- The inverse
- Both verification products
- Whether the byte is self-inverse
- The AES S-box output (inverse + affine transform)

In [ ]:
# Section 7 — Interactive inverse explorer

def aes_sbox(a):
    """Compute the AES S-box output for byte a."""
    inv = gf_inv(a) if a != 0 else 0
    # Affine transform
    b = inv
    result = 0
    c = 0x63
    for i in range(8):
        bit = ((b >> i) & 1) ^ ((b >> ((i+4)%8)) & 1) ^ \
              ((b >> ((i+5)%8)) & 1) ^ ((b >> ((i+6)%8)) & 1) ^ \
              ((b >> ((i+7)%8)) & 1) ^ ((c >> i) & 1)
        result |= (bit << i)
    return result

def gf_inv_explore(hex_val):
    """
    Explore the multiplicative inverse of a GF(2^8) element.

    Parameters
    ----------
    hex_val : str or int
        Byte value as a hex string (e.g. '0x53', '53') or integer (e.g. 0x53, 83).

    Example
    -------
    >>> gf_inv_explore('0x53')
    """
    if isinstance(hex_val, str):
        a = int(hex_val, 16)
    else:
        a = int(hex_val)

    if a < 0 or a > 255:
        print(f'Error: value {hex_val!r} is out of byte range (0–255).')
        return

    sep = '─' * 52
    print(sep)
    print(f'  Input byte : 0x{a:02X}  ({a:08b}b = {a})')

    if a == 0:
        print('  Inverse    : None  (0x00 has no multiplicative inverse)')
        print(f'  AES S-box  : 0x{aes_sbox(0):02X}  (0x00 maps to 0x63 by convention)')
        print(sep)
        return

    inv = gf_inv(a)
    prod_ab = gf_mul(a, inv)
    prod_ba = gf_mul(inv, a)
    is_self_inv = (inv == a)
    sbox_out = aes_sbox(a)

    print(f'  Inverse    : 0x{inv:02X}  ({inv:08b}b = {inv})')
    print(f'  0x{a:02X} × 0x{inv:02X} = 0x{prod_ab:02X}  (verify a·b = 1: {"✓" if prod_ab == 1 else "✗"})')
    print(f'  0x{inv:02X} × 0x{a:02X} = 0x{prod_ba:02X}  (verify b·a = 1: {"✓" if prod_ba == 1 else "✗"})')
    print(f'  Self-inverse: {is_self_inv}  {"(a^2 = 0x01)" if is_self_inv else ""}')
    print(f'  AES S-box  : 0x{sbox_out:02X}  (inverse → affine transform + 0x63)')
    print(sep)

# Example explorations
for val in ['0x02', '0x53', '0x01', '0xFF', '0x8D', '0x00']:
    gf_inv_explore(val)
    print()

## Section 8 — Special cases: 0x00, 0x01, and self-inverse bytes

### 0x00 — no inverse

Any element times zero is zero: `0x00 * b = 0x00 ≠ 0x01` for all `b`.  
So `0x00` is the unique element without an inverse.

### 0x01 — the multiplicative identity

`0x01 * b = b` for all `b`, so `0x01⁻¹ = 0x01`.  
It is the only byte that is trivially its own inverse.

### Self-inverse bytes (`a² = 0x01`)

A byte is *self-inverse* if `gf_mul(a, a) == 0x01`.  
We saw from Section 6 that exactly one such byte exists: `0x01` itself.  
We confirm and then explore why no others occur.

In [ ]:
# Section 8 — Special cases in detail

print('─── 0x00: the non-invertible element ───')
print(f'  gf_inv(0x00) = {gf_inv(0x00)}')
print(f'  gf_mul(0x00, 0x00) = 0x{gf_mul(0x00, 0x00):02X}')
print(f'  gf_mul(0x00, 0xFF) = 0x{gf_mul(0x00, 0xFF):02X}')
print(f'  In any ring, 0 * b = 0 ≠ 1, so no inverse exists.')
print()

print('─── 0x01: the multiplicative identity ───')
print(f'  gf_inv(0x01) = 0x{gf_inv(0x01):02X}')
print(f'  gf_mul(0x01, 0x01) = 0x{gf_mul(0x01, 0x01):02X}  (0x01 is self-inverse)')
print(f'  gf_mul(0x01, 0xAB) = 0x{gf_mul(0x01, 0xAB):02X}  (identity property: 1 * b = b)')
print()

print('─── Self-inverse bytes (a⁻¹ = a, i.e., a² = 0x01) ───')
self_inv_bytes = [a for a in range(1, 256) if gf_mul(a, a) == 1]
print(f'  Self-inverse bytes found: {len(self_inv_bytes)}')
for v in self_inv_bytes:
    print(f'  0x{v:02X}: gf_mul(0x{v:02X}, 0x{v:02X}) = 0x{gf_mul(v,v):02X}  ✓')
print()
print('  Algebraic note: a² = 1 means a is a root of x² - 1 = x² + 1 over GF(2).')
print('  In GF(2^8), x² + 1 = (x+1)² (since 2=0 in GF(2)), which has only one root: x = 1.')
print('  Therefore 0x01 is the only self-inverse byte in GF(2^8).')

## Section 9 — Connection to the AES S-box

The **AES SubBytes** operation (FIPS 197, Section 5.1.1) processes each byte of the state:

1. **Inversion in GF(2⁸):** replace `a` with `a⁻¹` (with `0x00 → 0x00`).
2. **Affine transform over GF(2):** apply the matrix–vector product and add constant `0x63`.

The affine transform is:

```
⎡ b0 ⎤   ⎡ 1 0 0 0 1 1 1 1 ⎤ ⎡ a0 ⎤   ⎡ 1 ⎤
⎢ b1 ⎥   ⎢ 1 1 0 0 0 1 1 1 ⎥ ⎢ a1 ⎥   ⎢ 1 ⎥
⎢ b2 ⎥   ⎢ 1 1 1 0 0 0 1 1 ⎥ ⎢ a2 ⎥   ⎢ 0 ⎥
⎢ b3 ⎥ = ⎢ 1 1 1 1 0 0 0 1 ⎥ ⎢ a3 ⎥ ⊕ ⎢ 0 ⎥
⎢ b4 ⎥   ⎢ 1 1 1 1 1 0 0 0 ⎥ ⎢ a4 ⎥   ⎢ 0 ⎥
⎢ b5 ⎥   ⎢ 0 1 1 1 1 1 0 0 ⎥ ⎢ a5 ⎥   ⎢ 1 ⎥
⎢ b6 ⎥   ⎢ 0 0 1 1 1 1 1 0 ⎥ ⎢ a6 ⎥   ⎢ 1 ⎥
⎣ b7 ⎦   ⎣ 0 0 0 1 1 1 1 1 ⎦ ⎣ a7 ⎦   ⎣ 0 ⎦
```

We now compute the full S-box table for a selection of bytes and display the results.

In [ ]:
# Section 9 — Full AES S-box construction and spot-checks

# Build the complete 256-entry S-box table
sbox = [aes_sbox(a) for a in range(256)]

# Spot-check against the published FIPS 197 S-box values
# Source: FIPS 197, Figure 7
known_sbox = {
    0x00: 0x63,
    0x01: 0x7C,
    0x02: 0x77,
    0x03: 0x7B,
    0x53: 0xED,
    0xF0: 0x8C,
    0xFF: 0x16,
    0x63: 0xFB,
    0x8D: 0x5D,
}

print('AES S-box spot-checks (vs FIPS 197 Figure 7):')
print(f'  {"Input":>6}  {"Our S-box":>10}  {"Expected":>10}  {"Match":>6}')
print('  ' + '-' * 38)
all_ok = True
for inp, expected in sorted(known_sbox.items()):
    our = sbox[inp]
    match = '✓' if our == expected else '✗'
    if our != expected:
        all_ok = False
    print(f'  0x{inp:02X}      0x{our:02X}        0x{expected:02X}       {match}')

print()
print(f'All spot-checks passed: {all_ok}')
print()

# Print the full 16x16 S-box table (as in FIPS 197 Figure 7)
print('Full AES S-box (hex), rows = high nibble, cols = low nibble:')
print('     ', end='')
for col in range(16):
    print(f' {col:X} ', end='')
print()
print('     ' + '---' * 16)
for row in range(16):
    print(f'  {row:X}  |', end='')
    for col in range(16):
        a = row * 16 + col
        print(f' {sbox[a]:02X}', end='')
    print()

## Section 10 — Summary table of sample inverse pairs

The table below lists 20 sample inverse pairs from across the byte range,
confirming the pattern and providing convenient reference values.

In [ ]:
# Section 10 — Summary table of selected inverse pairs

# Chosen to cover a variety of bit patterns and include several AES-notable values
sample_bytes = [
    0x01, 0x02, 0x03, 0x04, 0x08, 0x10, 0x20, 0x40, 0x80,  # powers of 2 and identity
    0x53, 0xCA,  # classic AES S-box worked example
    0x57, 0x13,  # Module 12 multiplication example
    0x8D, 0xF6,  # from common AES test vectors
    0x63, 0xFB,
    0xAE, 0x72,
    0xFF,
]

print(f'  {"Byte a":>8}  {"Inverse b":>10}  {"a · b":>8}  {"b · a":>8}  {"Verify":>7}')
print('  ' + '-' * 54)
for a in sample_bytes:
    b = gf_inv(a)
    if b is None:
        print(f'  0x{a:02X}       None           —         —       (no inverse)')
    else:
        ab = gf_mul(a, b)
        ba = gf_mul(b, a)
        check = '✓' if ab == 1 and ba == 1 else '✗'
        print(f'  0x{a:02X}       0x{b:02X}       0x{ab:02X}     0x{ba:02X}      {check}')

print()
print('Key takeaways:')
print('  1. Every nonzero byte has a unique inverse — GF(2^8) is a field.')
print('  2. Multiplication is commutative: a·b = b·a for all a, b.')
print('  3. The only self-inverse byte is 0x01 (the identity).')
print('  4. The map a ↦ a⁻¹ is an involution: (a⁻¹)⁻¹ = a.')
print('  5. The AES S-box first inverts in GF(2^8), then applies an affine transform.')